In [1]:
from google.colab import files
uploaded = files.upload()

Saving WorldCupMatches.csv to WorldCupMatches.csv
Saving WorldCupPlayers.csv to WorldCupPlayers.csv
Saving WorldCups.csv to WorldCups.csv


In [2]:
import pandas as pd

# Load the datasets
matches = pd.read_csv('WorldCupMatches.csv')
worldcups = pd.read_csv('WorldCups.csv')
players = pd.read_csv('WorldCupPlayers.csv')

# Let's see the first 5 rows of matches
print("=== MATCHES DATA ===")
print(matches.head())
print("\nShape:", matches.shape)

print("\n=== WORLD CUPS DATA ===")
print(worldcups.head())

print("\n=== COLUMNS IN MATCHES ===")
print(matches.columns.tolist())

=== MATCHES DATA ===
     Year              Datetime    Stage         Stadium         City  \
0  1930.0  13 Jul 1930 - 15:00   Group 1         Pocitos  Montevideo    
1  1930.0  13 Jul 1930 - 15:00   Group 4  Parque Central  Montevideo    
2  1930.0  14 Jul 1930 - 12:45   Group 2  Parque Central  Montevideo    
3  1930.0  14 Jul 1930 - 14:50   Group 3         Pocitos  Montevideo    
4  1930.0  15 Jul 1930 - 16:00   Group 1  Parque Central  Montevideo    

  Home Team Name  Home Team Goals  Away Team Goals Away Team Name  \
0         France              4.0              1.0         Mexico   
1            USA              3.0              0.0        Belgium   
2     Yugoslavia              2.0              1.0         Brazil   
3        Romania              3.0              1.0           Peru   
4      Argentina              1.0              0.0         France   

  Win conditions  Attendance  Half-time Home Goals  Half-time Away Goals  \
0                     4444.0                   3.

In [3]:
# Drop rows where team names or goals are missing
matches = matches.dropna(subset=['Home Team Name', 'Away Team Name', 'Home Team Goals', 'Away Team Goals'])

# Convert goals to integers (they were showing as 4.0, 1.0 etc)
matches['Home Team Goals'] = matches['Home Team Goals'].astype(int)
matches['Away Team Goals'] = matches['Away Team Goals'].astype(int)

# Create a new column called 'Result' - this is what our ML model will predict
# 'H' = Home team won, 'A' = Away team won, 'D' = Draw
def get_result(row):
    if row['Home Team Goals'] > row['Away Team Goals']:
        return 'H'
    elif row['Away Team Goals'] > row['Home Team Goals']:
        return 'A'
    else:
        return 'D'

matches['Result'] = matches.apply(get_result, axis=1)

# Check it worked
print(matches[['Home Team Name', 'Home Team Goals', 'Away Team Goals', 'Away Team Name', 'Result']].head(10))
print("\nResult value counts:")
print(matches['Result'].value_counts())

  Home Team Name  Home Team Goals  Away Team Goals Away Team Name Result
0         France                4                1         Mexico      H
1            USA                3                0        Belgium      H
2     Yugoslavia                2                1         Brazil      H
3        Romania                3                1           Peru      H
4      Argentina                1                0         France      H
5          Chile                3                0         Mexico      H
6     Yugoslavia                4                0        Bolivia      H
7            USA                3                0       Paraguay      H
8        Uruguay                1                0           Peru      H
9          Chile                1                0         France      H

Result value counts:
Result
H    488
D    190
A    174
Name: count, dtype: int64


In [4]:
# Calculate each team's historical stats - this becomes our model's "knowledge"
# For every team, we calculate their average goals scored and conceded across all matches

# Home stats per team
home_stats = matches.groupby('Home Team Name').agg(
    home_games=('Home Team Goals', 'count'),
    home_goals_scored=('Home Team Goals', 'mean'),
    home_goals_conceded=('Away Team Goals', 'mean')
).reset_index()
home_stats.columns = ['Team', 'home_games', 'home_goals_scored', 'home_goals_conceded']

# Away stats per team
away_stats = matches.groupby('Away Team Name').agg(
    away_games=('Away Team Goals', 'count'),
    away_goals_scored=('Away Team Goals', 'mean'),
    away_goals_conceded=('Home Team Goals', 'mean')
).reset_index()
away_stats.columns = ['Team', 'away_games', 'away_goals_scored', 'away_goals_conceded']

# Combine home and away stats into one table per team
team_stats = pd.merge(home_stats, away_stats, on='Team')
team_stats['avg_goals_scored'] = (team_stats['home_goals_scored'] + team_stats['away_goals_scored']) / 2
team_stats['avg_goals_conceded'] = (team_stats['home_goals_conceded'] + team_stats['away_goals_conceded']) / 2
team_stats['total_games'] = team_stats['home_games'] + team_stats['away_games']

print(team_stats[['Team', 'avg_goals_scored', 'avg_goals_conceded', 'total_games']].sort_values('avg_goals_scored', ascending=False).head(10))

                          Team  avg_goals_scored  avg_goals_conceded  \
66                      Turkey          3.125000            1.062500   
31                     Hungary          2.527778            1.884921   
25                     Germany          2.264706            0.970588   
26                  Germany FR          1.993268            1.497552   
7                       Brazil          1.962946            1.167917   
52                      Russia          1.833333            1.250000   
18              Czechoslovakia          1.775000            1.325000   
23                      France          1.763441            1.183333   
73  rn">Bosnia and Herzegovina          1.750000            1.250000   
20                     Denmark          1.706349            1.539683   

    total_games  
66           10  
31           32  
25           48  
26           62  
7           108  
52            9  
18           30  
23           61  
73            3  
20           16  


In [5]:
# Fix the Germany split - merge East/West Germany into one
matches['Home Team Name'] = matches['Home Team Name'].replace('Germany FR', 'Germany')
matches['Away Team Name'] = matches['Away Team Name'].replace('Germany FR', 'Germany')

# Recalculate team stats after the fix
home_stats = matches.groupby('Home Team Name').agg(
    home_games=('Home Team Goals', 'count'),
    home_goals_scored=('Home Team Goals', 'mean'),
    home_goals_conceded=('Away Team Goals', 'mean')
).reset_index()
home_stats.columns = ['Team', 'home_games', 'home_goals_scored', 'home_goals_conceded']

away_stats = matches.groupby('Away Team Name').agg(
    away_games=('Away Team Goals', 'count'),
    away_goals_scored=('Away Team Goals', 'mean'),
    away_goals_conceded=('Home Team Goals', 'mean')
).reset_index()
away_stats.columns = ['Team', 'away_games', 'away_goals_scored', 'away_goals_conceded']

team_stats = pd.merge(home_stats, away_stats, on='Team')
team_stats['avg_goals_scored'] = (team_stats['home_goals_scored'] + team_stats['away_goals_scored']) / 2
team_stats['avg_goals_conceded'] = (team_stats['home_goals_conceded'] + team_stats['away_goals_conceded']) / 2
team_stats['total_games'] = team_stats['home_games'] + team_stats['away_games']

# Now attach team stats to each match
# For each match, we pull stats for BOTH teams and put them side by side
df = matches[['Home Team Name', 'Away Team Name', 'Result']].copy()

df = df.merge(team_stats[['Team', 'avg_goals_scored', 'avg_goals_conceded', 'total_games']],
              left_on='Home Team Name', right_on='Team').drop('Team', axis=1)
df = df.rename(columns={
    'avg_goals_scored': 'home_avg_scored',
    'avg_goals_conceded': 'home_avg_conceded',
    'total_games': 'home_total_games'
})

df = df.merge(team_stats[['Team', 'avg_goals_scored', 'avg_goals_conceded', 'total_games']],
              left_on='Away Team Name', right_on='Team').drop('Team', axis=1)
df = df.rename(columns={
    'avg_goals_scored': 'away_avg_scored',
    'avg_goals_conceded': 'away_avg_conceded',
    'total_games': 'away_total_games'
})

print(df.head())
print("\nShape:", df.shape)
print("\nAny missing values?", df.isnull().sum().sum())

  Home Team Name Away Team Name Result  home_avg_scored  home_avg_conceded  \
0         France         Mexico      H         1.763441           1.183333   
1            USA        Belgium      H         1.133333           1.831579   
2     Yugoslavia         Brazil      H         1.685294           1.189706   
3        Romania           Peru      H         1.458333           1.513889   
4      Argentina         France      H         1.435185           1.166667   

   home_total_games  away_avg_scored  away_avg_conceded  away_total_games  
0                61         1.161184           1.435855                54  
1                34         1.290000           1.484444                43  
2                37         1.962946           1.167917               108  
3                21         1.600000           1.750000                15  
4                81         1.763441           1.183333                61  

Shape: (835, 9)

Any missing values? 0


In [6]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

# Define our features (inputs) and target (what we want to predict)
features = ['home_avg_scored', 'home_avg_conceded', 'home_total_games',
            'away_avg_scored', 'away_avg_conceded', 'away_total_games']

X = df[features]  # inputs
y = df['Result']  # output we want to predict (H, A, or D)

# Split data - 80% for training the model, 20% for testing it
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Build and train the model
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# Test the model
y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print(f"Model Accuracy: {accuracy:.2%}")
print("\nDetailed Report:")
print(classification_report(y_test, y_pred))

Model Accuracy: 54.49%

Detailed Report:
              precision    recall  f1-score   support

           A       0.54      0.48      0.51        29
           D       0.19      0.17      0.18        36
           H       0.65      0.70      0.67       102

    accuracy                           0.54       167
   macro avg       0.46      0.45      0.45       167
weighted avg       0.53      0.54      0.54       167



In [7]:
def predict_match(home_team, away_team):
    # Check if both teams exist in our dataset
    if home_team not in team_stats['Team'].values:
        print(f"'{home_team}' not found in dataset. Check spelling.")
        return
    if away_team not in team_stats['Team'].values:
        print(f"'{away_team}' not found in dataset. Check spelling.")
        return

    # Pull stats for both teams
    home = team_stats[team_stats['Team'] == home_team].iloc[0]
    away = team_stats[team_stats['Team'] == away_team].iloc[0]

    # Build the input row for the model
    input_data = pd.DataFrame([{
        'home_avg_scored': home['avg_goals_scored'],
        'home_avg_conceded': home['avg_goals_conceded'],
        'home_total_games': home['total_games'],
        'away_avg_scored': away['avg_goals_scored'],
        'away_avg_conceded': away['avg_goals_conceded'],
        'away_total_games': away['total_games']
    }])

    # Get prediction and probabilities
    prediction = model.predict(input_data)[0]
    probabilities = model.predict_proba(input_data)[0]
    classes = model.classes_

    result_map = {'H': f'{home_team} wins', 'A': f'{away_team} wins', 'D': 'Draw'}

    print(f"\n{'='*45}")
    print(f"  {home_team} vs {away_team}")
    print(f"{'='*45}")
    print(f"  Predicted Result: {result_map[prediction]}")
    print(f"\n  Probabilities:")
    for cls, prob in zip(classes, probabilities):
        print(f"    {result_map[cls]}: {prob:.1%}")
    print(f"{'='*45}")

# Try some WC 2026 matches!
predict_match('Brazil', 'Argentina')
predict_match('France', 'Germany')
predict_match('USA', 'Mexico')
predict_match('Portugal', 'Argentina')


  Brazil vs Argentina
  Predicted Result: Brazil wins

  Probabilities:
    Argentina wins: 2.4%
    Draw: 39.2%
    Brazil wins: 58.4%

  France vs Germany
  Predicted Result: Germany wins

  Probabilities:
    Germany wins: 72.9%
    Draw: 1.0%
    France wins: 26.1%

  USA vs Mexico
  Predicted Result: USA wins

  Probabilities:
    Mexico wins: 34.3%
    Draw: 12.4%
    USA wins: 53.3%

  Portugal vs Argentina
  Predicted Result: Portugal wins

  Probabilities:
    Argentina wins: 21.0%
    Draw: 8.6%
    Portugal wins: 70.4%


In [8]:
# Generate predictions for a bunch of WC 2026 matchups
matchups = [
    ('Brazil', 'Argentina'),
    ('France', 'Germany'),
    ('USA', 'Mexico'),
    ('Portugal', 'Argentina'),
    ('Brazil', 'France'),
    ('Germany', 'Argentina'),
    ('Spain', 'Brazil'),
    ('England', 'France'),
    ('Netherlands', 'Portugal'),
    ('USA', 'England'),
    ('Brazil', 'Germany'),
    ('Spain', 'France'),
    ('Argentina', 'France'),
    ('Portugal', 'Brazil'),
    ('Mexico', 'Argentina'),
    ('USA', 'Germany'),
    ('Spain', 'Germany'),
    ('England', 'Brazil'),
]

results = []
for home, away in matchups:
    if home not in team_stats['Team'].values or away not in team_stats['Team'].values:
        continue

    home_data = team_stats[team_stats['Team'] == home].iloc[0]
    away_data = team_stats[team_stats['Team'] == away].iloc[0]

    input_data = pd.DataFrame([{
        'home_avg_scored': home_data['avg_goals_scored'],
        'home_avg_conceded': home_data['avg_goals_conceded'],
        'home_total_games': home_data['total_games'],
        'away_avg_scored': away_data['avg_goals_scored'],
        'away_avg_conceded': away_data['avg_goals_conceded'],
        'away_total_games': away_data['total_games']
    }])

    prediction = model.predict(input_data)[0]
    probabilities = model.predict_proba(input_data)[0]
    classes = list(model.classes_)

    prob_dict = dict(zip(classes, probabilities))

    results.append({
        'Home Team': home,
        'Away Team': away,
        'Predicted Result': prediction,
        'Home Win %': round(prob_dict.get('H', 0) * 100, 1),
        'Away Win %': round(prob_dict.get('A', 0) * 100, 1),
        'Draw %': round(prob_dict.get('D', 0) * 100, 1),
    })

predictions_df = pd.DataFrame(results)
print(predictions_df)

# Export to CSV
predictions_df.to_csv('wc2026_predictions.csv', index=False)
print("\n✅ Predictions saved!")

# Also export team stats
team_stats_export = team_stats[['Team', 'avg_goals_scored', 'avg_goals_conceded', 'total_games']].copy()
team_stats_export.columns = ['Team', 'Avg Goals Scored', 'Avg Goals Conceded', 'Total WC Games']
team_stats_export.to_csv('team_stats.csv', index=False)
print("✅ Team stats saved!")

# Also export the cleaned matches data
matches[['Year', 'Home Team Name', 'Away Team Name', 'Home Team Goals', 'Away Team Goals', 'Result', 'Stage']].to_csv('matches_clean.csv', index=False)
print("✅ Clean matches saved!")

      Home Team  Away Team Predicted Result  Home Win %  Away Win %  Draw %
0        Brazil  Argentina                H        58.4         2.4    39.2
1        France    Germany                A        26.1        72.9     1.0
2           USA     Mexico                H        53.3        34.3    12.4
3      Portugal  Argentina                H        70.4        21.0     8.6
4        Brazil     France                A        25.6        46.0    28.3
5       Germany  Argentina                D        48.8         0.5    50.7
6         Spain     Brazil                A        47.7        47.9     4.4
7       England     France                H        95.4         1.0     3.6
8   Netherlands   Portugal                H        70.3         5.4    24.3
9           USA    England                H        75.0        11.5    13.5
10       Brazil    Germany                A        23.8        73.5     2.7
11        Spain     France                H        52.5        34.5    13.0
12    Argent

In [9]:
files.download('wc2026_predictions.csv')
files.download('team_stats.csv')
files.download('matches_clean.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [16]:
# Proper 16 team single elimination bracket - clean and simple
# Taking the best 16 teams from our group stage

sweet_16 = [
    # Left side of bracket
    ('Brazil', 'Mexico'),
    ('Germany', 'Netherlands'),
    ('France', 'England'),
    ('Argentina', 'Portugal'),
    # Right side of bracket
    ('Spain', 'Belgium'),
    ('Turkey', 'Switzerland'),
    ('Morocco', 'Senegal'),
    ('Uruguay', 'Croatia'),
]

def run_round(matchups, round_name):
    print("\n" + "=" * 50)
    print(f"          {round_name}")
    print("=" * 50)
    winners = []
    for home, away in matchups:
        winner = predict_winner_v2(home, away)
        print(f"  {home} vs {away} → {winner} ✅")
        winners.append(winner)
    return winners

# Round of 16
qf_teams = run_round(sweet_16, "ROUND OF 16")

# Quarter Finals
qf_matchups = [(qf_teams[i], qf_teams[i+1]) for i in range(0, len(qf_teams), 2)]
sf_teams = run_round(qf_matchups, "QUARTER FINALS")

# Semi Finals
sf_matchups = [(sf_teams[i], sf_teams[i+1]) for i in range(0, len(sf_teams), 2)]
finalists = run_round(sf_matchups, "SEMI FINALS")

# Final
print("\n" + "=" * 50)
print("              FINAL 🏆")
print("=" * 50)
champion = predict_winner_v2(finalists[0], finalists[1])
print(f"\n  {finalists[0]} vs {finalists[1]}")
print(f"\n  🏆 FIFA WC 2026 WINNER: {champion} 🏆")
print("=" * 50)


          ROUND OF 16
  Brazil vs Mexico → Brazil ✅
  Germany vs Netherlands → Germany ✅
  France vs England → France ✅
  Argentina vs Portugal → Argentina ✅
  Spain vs Belgium → Spain ✅
  Turkey vs Switzerland → Turkey ✅
  Morocco vs Senegal → Morocco ✅
  Uruguay vs Croatia → Croatia ✅

          QUARTER FINALS
  Brazil vs Germany → Germany ✅
  France vs Argentina → France ✅
  Spain vs Turkey → Spain ✅
  Morocco vs Croatia → Croatia ✅

          SEMI FINALS
  Germany vs France → Germany ✅
  Spain vs Croatia → Croatia ✅

              FINAL 🏆

  Germany vs Croatia

  🏆 FIFA WC 2026 WINNER: Croatia 🏆
